In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("SPTRANS_TOKEN")

BASE_URL = "https://api.olhovivo.sptrans.com.br/v2.1"

print("Token encontrado:", TOKEN is not None)

In [ ]:
sessao = requests.Session()

resposta = sessao.post(
    f"{BASE_URL}/Login/Autenticar",
    params={"token": TOKEN},
    timeout=15
)

print("Código HTTP:", resposta.status_code)
print("Resposta:", resposta.text)

In [ ]:
resposta = sessao.get(f"{BASE_URL}/Linha/Buscar", params={"termosBusca": "8000"}, timeout=15) # Na variável resposta, vou pegar(Get) dentro da Base URl, o end-point 
linhas = resposta.json()  # transforma o texto da resposta em lista do Python

tabela_linhas = pd.DataFrame(linhas) # Peguei a variavel LINHAS e transformei esse resultado em um DataFrame para se tornar uma tabela.
tabela_linhas

In [ ]:
codigo_linha = 35274

resposta = sessao.get(f"{BASE_URL}/Parada/BuscarParadasPorLinha", # Chamando(Get) End-point da Base_URL para retornar informações de Paradas das Linhas.
                      params={"codigoLinha": codigo_linha}, timeout=15) # Passei como parametro que a resposta deve receber informações guardadas no código da variável codigo_linha.
paradas = pd.DataFrame(resposta.json()) # Estou transformando a resposta que veio em HTTPS em uma lista Json (Dicionário) e trasformando o Json em uma "DF" de Tabela, guardando em uma variável.

paradas = paradas.rename(columns={   # Utilizando o ".rename", eu tratei nomes das colunas para ficar mais prático de análisar olhando os nomes.
    "cp": "codigo_parada",
    "np": "nome_parada",
    "ed": "endereco",
    "py": "latitude",
    "px": "longitude"
})

print("Quantidade de paradas:", len(paradas)) #Linha Básica para verificar o tamanho da tabela paradas utilzando "Len"
paradas.head(15)  # Por tendência o ".head" pega as 5 primeiras linhas da tabela, mas podemos passar no parametro quantas linhas podemos ver.

In [ ]:
from math import radians, sin, cos, asin, sqrt # Importei da biblioteca Math, algumas operações matemáticas que vou precisar.

def distancia_m(lat1, lon1, lat2, lon2):
    """Distância em metros entre dois pontos (fórmula de Haversine)."""
    Raio = 6371000  # O numero atribuído a variável representa o raio da Terra em metros
    dlat = radians(lat2 - lat1) # Guarda o resultado passado para Radianos da conta de (Lat2 - Lat1) que está cotada em Graus.
    dlon = radians(lon2 - lon1) # Guarda o resultado passado para Radianos da conta de (Lon2 - Lon1) que está cotada em Graus.
    
    
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * Raio * asin(sqrt(a))

In [ ]:

distancias = [] # Criando uma variável de lista para guardar as distancias
for i in range(1, len(paradas)): # Loop para pegar cada Parada da tabela Paradas
    anterior = paradas.iloc[i - 1] # Guardando o valor anterior do valor atual (i) do loop na variável "anterior".
    atual = paradas.iloc[i] # Guardando o valor atual do Loop (i) na variável "atual"
    distancias.append( # O append aqui serve para guardar na lista "distancia".
        distancia_m(anterior["latitude"], anterior["longitude"], # Dentro do append, chamei a função para calcular os parametros recebidos da latitude e longitude das variaveis "anterior" e "atual".
                    atual["latitude"], atual["longitude"])
    )

serie = pd.Series(distancias) # Guardei em uma variável e tranformei a lista "distancia" em uma Série com indices e valores para ánalisa melhor.
print(serie.describe()) # O describe serve para fazer uma análise Estatística sobre a Série "Distancias"
print("Saltos maiores que 1000 m:", (serie > 1000).sum(), "de", len(serie)) # Retornei utilizando um filtro, a contagem de distancias de são > 1000 e comparei com o tamanho da lista utilizando "Len".


In [ ]:
def paradas_proximas(tabela_paradas, lat, lon, raio_max): # Essa função serve para encontrar as paradas mais próximas do ponto de origem.
    """Devolve as paradas a até raio_max metros do ponto, da mais perto para a mais longe."""
    
    tabela = tabela_paradas.copy() # Copia o DataFrame original e salva em outra variável "teste", para não alterar o DataFrame Original.
    tabela["distancia_m"] = tabela.apply( # Estou criando uma coluna chamada "distancia_m" na nova tabela e estou aplicando uma Função com o apply.
        lambda parada: distancia_m(lat, lon, parada["latitude"], parada["longitude"]), axis=1 # Lamba é uma mini função dentro do apply que vai chamar a função distancia_m e passar os parametros das cordenadas do usuário e das paradas.
    ) # o "Axis = 1" significa que vai trabalhar o lambda pegando linha por linha, se fosse "0" seria para pegar de coluna em coluna
    
    return tabela[tabela["distancia_m"] <= raio_max].sort_values("distancia_m") # Aplicando um filtro para retornar uma tabela com apenas as distâncias que atendem o parametro de "Raio_max", e ordenando esses valores de forma ascentente.

## Importando Pasta GTFS para melhorar e filtrar melhores buscas de Paradas, Viagens e Linhas.

    Percebi que somente com as buscas da API Olho Vivo, estavam retornando informações do tráfego de forma desorganizada, para isso, resolvi buscar na mesma documentação do site uma forma de retornar essas buscas mais estruturadas, e encontrei o GTFS, que funciona como um guia oficial de como as linhas são desenhadas, Ou seja, enquanto a API retornava somente poucas linhas, ordem "aleatória" e sem mapa de trajeto, O GTFS vai nos complementar essa informação de forma coletiva com  a API Olho vivo.



In [ ]:
import os

PASTA_GTFS = "../data/_sptrans_gtfs"
print(os.listdir(PASTA_GTFS))

In [ ]:
# Aqui eu estou carregando os arquivos GTFS

routes = pd.read_csv(f"{PASTA_GTFS}/routes.txt", dtype=str)
trips = pd.read_csv(f"{PASTA_GTFS}/trips.txt", dtype=str)
stops = pd.read_csv(f"{PASTA_GTFS}/stops.txt", dtype=str)
stop_times = pd.read_csv(f"{PASTA_GTFS}/stop_times.txt", dtype=str)

# Conferindo o conteúdo de cada tabela

for nome, tabela in [
    ("routes", routes),
    ("trips", trips),
    ("stops", stops),
    ("stop_times", stop_times)
]:
    print(f"\n{nome}")
    print("Quantidade de registros:", len(tabela))
    print("Colunas:", tabela.columns.tolist())


In [ ]:

# Verificando a estrutura e alguns registros das tabelas

print("ROTAS")
display(routes.head(3))

print("VIAGENS")
display(trips.head(3))

print("PARADAS")
display(stops.head(3))

print("SEQUÊNCIA DAS PARADAS")
display(stop_times.head(3))


In [ ]:

# Verificando os relacionamentos entre as tabelas GTFS:
# Estou verificando a inexistência dos requistos que pedi
# com o sinal de "~", esse sinal reprepresanta a inversão da verdade, ou seja, quero ver quantos numeros do que pedi
# estão sem um relacionamento, para isso utilizei a inversão de informação "~".

print("Viagens com route_id inexistente:")
print((~trips["route_id"].isin(routes["route_id"])).sum()) # Contando (.sum) se route_id está dentro de routes([route_id])  e invertendo com ~ para descobrir se existe algum relacionamento vazio

print("\nRegistros de stop_times com trip_id inexistente:") 
print((~stop_times["trip_id"].isin(trips["trip_id"])).sum()) # Mesma relação com o comentário acima, porém com colunas diferentes.

print("\nRegistros de stop_times com stop_id inexistente:")
print((~stop_times["stop_id"].isin(stops["stop_id"])).sum()) # Mesma relação com o comentário acima, porém com colunas diferentes.


## Relacionamento das tabelas

    Aqui eu inicio um processo interessante e importante para a conexão das informações (Tabelas), Nessa etapa do código, vou utilizar o conceito de Modelagem de Dados para relacionar as tabelas entre si.

In [ ]:

# Fazendo o  1° relacionamento de viagens com suas respectivas linhas 

# Aplicando o conceito de modelagem de dados - Left Join

viagens_linhas = trips.merge( # Merge - Faz a mistura das duas tabelas
    routes,
    on="route_id", # Informa qual coluna irá conectar as tabelas
    how="left" # Conecta as informações mas mantém todos os registros da tabela "trips" a esquerda.
)

display(viagens_linhas.head()) # Apresentando como ficou a relação com as tabelas que foram conectadas.


In [ ]:
# 2° Relacionamento

# Adicionando as paradas de cada viagem

viagens_paradas = viagens_linhas.merge( # Relacionando as paradas de cada viagem
    stop_times,
    on="trip_id", # Conectando as informações pela referência ("trip_id")
    how="left" # Mantendo as informações da tabela "viagens_linhas" a Esquerda
)

display(viagens_paradas.head())


In [ ]:
# 3° Ultimo relacionamento - Estrutura Geral GTFS

# Adicionando nome e coordenadas dos pontos

gtfs_completo = viagens_paradas.merge( # Essa varaiável irá guardar todas as informações das relações entre tabelas que fizemos.
    stops, # Tabela paradas quue irá se relacionar com a tabela acima.
    on="stop_id", # As informações vão se conectar pela coluna de referencia "stop_id"
    how="left" # A tabela viagens_paradas irá manter as informações a esquerda, enquanto a tabela stops ficará a direita de acordo com a relação.
)

display(gtfs_completo.head())


In [ ]:

# Conferindo a tabela integrada

print("Quantidade de registros:", len(gtfs_completo))
print("\nColunas disponíveis:")
print(gtfs_completo.columns.tolist())

print("\nInformações da tabela:")
gtfs_completo.info()


In [ ]:
# Seleciona o primeiro trip_id disponível
viagem_exemplo = gtfs_completo["trip_id"].iloc[0]

print("Viagem selecionada:", viagem_exemplo)

# Atribuíndo valor ás variáveis

paradas_viagem = gtfs_completo[ # Essa variável só irá guardar as informações do DataFrame gtfs_compelto
    gtfs_completo["trip_id"] == viagem_exemplo # Guarda todos os id_linha do DataFrame que for == "viagem exemplo"
].copy() # Copiando o DataFrame "gtfs_completo" com a filtragem para não alterar o DataFrame original

paradas_viagem["stop_sequence"] = pd.to_numeric( # Alterando as sequncia de parada para tipo numero
    paradas_viagem["stop_sequence"]
)

paradas_viagem = paradas_viagem.sort_values( # Ordenando os valores (stop_sequence) e ordem ascendente.
    "stop_sequence"
)

display( # Aqui estou mostrando quais informações (Colunas) quero que retorne na minha busca.
    paradas_viagem[
        [
            "trip_id",
            "stop_sequence",
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon"
        ]
    ]
)

## Foco dessa etapa: Diagnostico das linhas dos Ônibus

    Na etapa passada pude verificar e fazer a relação entre as informações que recolhi do GTFS, agora, a minha ideia é analisar a correlação dessas informações sobres as linhas, Viagens, Paradas e etc... 

In [ ]:
viagens_por_linha = ( # Guardando em uma variável a filtragem do algoritimo abaixo
    trips.groupby("route_id")["trip_id"] # Groupyby utilizado para agrupar as informações pelos identificadores passados. Trip_id é a coluna onde queremos analisar
    .nunique() # conta quantas viagem diferentes possui em cada linha
    .reset_index(name="quantidade_viagens") # Pega a filtragem e tranforma em uma tabela organizada, dando um nome para a coluna  a ser analisada.
)

display(viagens_por_linha.head(10))

In [ ]:
paradas_por_linha = (
    gtfs_completo.groupby("route_id")["stop_id"] # Agrupando pelo route_id dando como referência de busca a coluna "stop_id"
    .nunique() # Contando quantas paradas diferentes possuem em cada linha agrupada
    .reset_index(name="quantidade_paradas") # Ordenando os valores em uma tabela organizada e atribuindo um nome para a coluna
)

display(paradas_por_linha.head(10))

In [ ]:
resumo_linhas = ( # Fazendo um Merge de tabelas para resumir informações
    routes[["route_id", "route_short_name", "route_long_name"]] # Se referindo as colunas da tabela routes
    .merge(viagens_por_linha, on="route_id", how="left") # Relacionando os route_id que aparecem nas duas tabelas e relacionando essas informações mantendo a primeira tabela a esquerda.
    .merge(paradas_por_linha, on="route_id", how="left") # Faz um merge relacionado ao merge anterior e pega como referência o route_id para juntar as informações.
)

display(resumo_linhas.head(10))

In [ ]:
ranking_linhas = resumo_linhas.sort_values( # Variavel que irá rankear o algoritimo abaixo
    by="quantidade_viagens", # Selecionando a coluna ("quantidade_viagens") da variável resumo_linhas
    ascending=False # Ordenando de forma descentente as informações.
)

display(ranking_linhas.head(10)) # Retornando as 10 primeiras linhas da variável

In [ ]:
linhas_sem_viagens = resumo_linhas[
    resumo_linhas["quantidade_viagens"].isna() # Buscando somente os registros na coluna quantidade_viagens que possuem valor nulo
]

display(linhas_sem_viagens) # Mostrando as linhas guardadas como nulas da busca acima

## Identificação de paradas próximas á origem e ao destino

    O foco nessa etapa será

In [ ]:
# Buscando colunas de uma variável e guardando em outra variável para realizar uma analise sem alterar a variável com infos importantes.
paradas_gtfs = stops[
    ["stop_id", "stop_name", "stop_lat", "stop_lon"]
].copy()

#Alterando o nome das colunas para um nome intuítivo
paradas_gtfs = paradas_gtfs.rename(columns={
    "stop_lat": "latitude",
    "stop_lon": "longitude"
})

# Tranformando o tipo de dado das colunas e convertendo para Número
paradas_gtfs["latitude"] = pd.to_numeric(paradas_gtfs["latitude"])
paradas_gtfs["longitude"] = pd.to_numeric(paradas_gtfs["longitude"])

## Validando Cordenadas para Teste de algoritmo.


In [ ]:
# Coordenadas de exemplo para o primeiro teste do algoritmo
origem = {
    "latitude": -23.5505,
    "longitude": -46.6333
}

destino = {
    "latitude": -23.5614,
    "longitude": -46.6559
}

raio_busca = 500

## Chamando função e guardando informações em Variáveis

In [ ]:
# Chamando função que está recebendo parametros de Origem e guardando o resultado em uma variável
paradas_origem = paradas_proximas(
    paradas_gtfs,
    origem["latitude"],
    origem["longitude"],
    raio_busca
)

# Chamando função que está recebendo parametros de destino e guardando os resultados em uma variável
paradas_destino = paradas_proximas(
    paradas_gtfs,
    destino["latitude"],
    destino["longitude"],
    raio_busca
)

# Chamando as variáveis que guardaram os resultados da função
print("Paradas próximas da origem:", len(paradas_origem))
print("Paradas próximas do destino:", len(paradas_destino))

## Transformando e filtrando informações

In [ ]:
# Bloco utilizado para identificar e guardar quais linhas passam próximas á origem que o passageiro está.
linhas_origem = set( # O set é usado para tranformar informações e um conjunto sem duplicadas, ou seja, não repetir as linhas retornadas.
    gtfs_completo[
        gtfs_completo["stop_id"].isin(paradas_origem["stop_id"]) # Verificando quais valores da tabela gtfs_completo está na tabela parada origem.
    ]["route_id"].unique() # Retornando valores unicos da coluna route_id.
)

# Bloco utilizado para verificar quais linhas passam próximas ao destino, sem haver linhas duplicadas.
linhas_destino = set(
    gtfs_completo[
        gtfs_completo["stop_id"].isin(paradas_destino["stop_id"])
    ]["route_id"].unique()
)

linhas_candidatas = linhas_origem.intersection(linhas_destino) # Verificando e guardando a intersseção de identificadores nas variáveis.

print("Linhas próximas da origem:", len(linhas_origem))
print("Linhas próximas do destino:", len(linhas_destino))
print("Linhas em comum:", len(linhas_candidatas))

## Identificação e classificação de viagens diretas

In [ ]:
# Identificando as paradas de embarque
embarques = gtfs_completo[ 
    gtfs_completo["stop_id"].isin(paradas_origem["stop_id"]) # Verificando quais valores da tabela gtfs_completo está na tabela parada origem.
][["route_id", "trip_id", "stop_id", "stop_sequence"]].copy() # Aqui eu estou filtrando quais são as colunas que me interessam e estou guardando em uma variável para não alterar o DF original.

# Identificando as paradas de desembarque
desembarques = gtfs_completo[
    gtfs_completo["stop_id"].isin(paradas_destino["stop_id"]) # Verificando quais valores da tabela gtfs_completo está na tabela parada destino.
]
][["route_id", "trip_id", "stop_id", "stop_sequence"]].copy() 

# Nesse bloco eu estou convertendo as colunas para número, futuramente será utilizada para comparações numéricas.
embarques["stop_sequence"] = pd.to_numeric(embarques["stop_sequence"])
desembarques["stop_sequence"] = pd.to_numeric(desembarques["stop_sequence"])

# Relacionando embarques e desembarques da mesma viagem
viagens_diretas = embarques.merge( # Estou relacioando as informações que se referem a mesma viagem.
    desembarques,
    on=["route_id", "trip_id"], # Nessa linha, mantive as informações que se referem a mesma viagem, no caso, os mesmo identidicadores de linha e viagem.
    suffixes=("_origem", "_destino") # Separei as informações de embarque e desembarque com sufixos para não confundir as informações que se referem a mesma viagem.
)

# Mantendo apenas viagens em que o destino vem depois da origem, ou seja, quero manter apenas viagens que o stop_sequence destino é maior que o de origem.
viagens_diretas = viagens_diretas[
    viagens_diretas["stop_sequence_destino"] > # Fazendo a comparação entre as colunas stop_sequence_destino e stop_sequence_origem, para manter apenas viagens que o destino vem depois da origem.
    viagens_diretas["stop_sequence_origem"]
]

display(viagens_diretas.head(10))

# Identificação de paradas Embarque/Desembarque

In [ ]:
## Identificação de parada de embarque

# Nesse bloco estou fazendo um merge entre a tabela de viagens_diretas e a tabela paradas_origem, para trazer informações de nome da parada e distância da parada de embarque.
opcoes_viagem = viagens_diretas.merge(
    paradas_origem[["stop_id", "stop_name", "distancia_m"]],
    left_on="stop_id_origem", # o left on serve para relacionar qual coluna da tabela da esquerda vai se relacionar com a tabela da direita.
    right_on="stop_id", # O right on serve para relacionar qual informação da coluna da direita vai se relacionar com a coluna da esquerda.
    how="left" # Vou manter as informações da tabela da esquerda.
)

# Renomeando colunas para nomes mais entendiveis e intuitivos.
opcoes_viagem = opcoes_viagem.rename(columns={
    "stop_name": "parada_embarque",
    "distancia_m": "distancia_origem"
})

opcoes_viagem = opcoes_viagem.drop(columns="stop_id") # Apagando a coluna stop_id que não é mais útil.

display(opcoes_viagem.head())

In [ ]:
## Identificação de parada de desembarque

opcoes_viagem = opcoes_viagem.merge(
    paradas_destino[["stop_id", "stop_name", "distancia_m"]], # Filtrando as colunas que quero da tabela de destino.
    left_on="stop_id_destino", # Pegando a coluna que quero usar de referencia da tabela da esquerda.
    right_on="stop_id", # Pegando a coluna que quero usar de referencia da tabela da direita.
    how="left" # Vou manter as informações da tabela da esquerda.
)

opcoes_viagem = opcoes_viagem.rename(columns={ # Renomeando colunas para nomes mais entendiveis e intuitivos.
    "stop_name": "parada_desembarque",
    "distancia_m": "distancia_destino"
})

opcoes_viagem = opcoes_viagem.drop(columns="stop_id") # Excluindo a coluna stop_id, pois usamos ela acima somente para fazer a comparação.

display(opcoes_viagem.head())

In [ ]:
opcoes_viagem = opcoes_viagem.merge( # Pegando informações da tabela de rotas e relacionando com a tabela opções de viagem.
    routes[["route_id", "route_short_name", "route_long_name"]], # Selecionando quais são as colunas que quero olhar da tabela rotas.
    on="route_id", # Usando como referencia a coluna route_id para conseguir relacionar as informações das tabelas.
    how="left" # Mantendo as informações da tabela da esquerda.(opções_viagem)
)

opcoes_viagem = opcoes_viagem.rename(columns={ # Renomeando colunas.
    "route_short_name": "numero_linha",
    "route_long_name": "nome_linha"
})

display(opcoes_viagem.head())

In [ ]:
opcoes_viagem["distancia_total"] = ( # Criando uma nova coluna que irá guardar a soma das distâncias de origem e destino.
    opcoes_viagem["distancia_origem"] + opcoes_viagem["distancia_destino"] # Somando as informalções das colunas.
)

opcoes_viagem = opcoes_viagem.sort_values( # Ordenando de forma crescente a tabela, tendo como referencia a coluna distancia_total. Priorizando as viagens que possuem menor distância total.
    by="distancia_total"
)

display(opcoes_viagem.head(10))